# DenseNet121 V2: Optimized Mammogram Classification

## CBIS-DDSM ROI-Based Breast Cancer Detection

### Architecture Overview

This notebook implements an optimized DenseNet121 model for binary classification of mammographic ROI images into BENIGN and MALIGNANT categories.

### Key Optimizations from V1/V12 Analysis

| Component | V12 Baseline | V2 Optimized | Rationale |
|-----------|--------------|--------------|----------|
| Preprocessing | Custom CLAHE | DenseNet `preprocess_input` | Proper ImageNet alignment |
| Dense Units | 2048 | 2048 | Maintain capacity |
| Dropout | 0.35 | 0.4 (staged) | Reduce overfitting |
| Batch Norm | After Dense | Before Activation | Standard practice |
| LR Schedule | ReduceLROnPlateau | Cosine Annealing + Warmup | Smoother convergence |
| Loss | Focal Loss | Focal + Label Smoothing | Better calibration |

### Computational Requirements

- **GPU**: NVIDIA A100 (40GB VRAM)
- **Runtime**: Google Colab Pro+
- **Dataset**: CBIS-DDSM ROI images (preprocessed cache)

### References

1. Huang et al. (2017). Densely Connected Convolutional Networks. CVPR.
2. Keras Applications: DenseNet preprocessing uses `mode='torch'` (scale to [-1, 1]).

In [ ]:
# =============================================================================
# Cell 1: Environment Setup
# =============================================================================

import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

# Detect runtime environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted")
except ImportError:
    IN_COLAB = False
    print("Local environment detected")

# GPU verification
import subprocess
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
    ).decode('utf-8').strip()
    print(f"GPU: {gpu_info}")
except Exception:
    print("No GPU detected")

RANDOM_SEED = 42

In [ ]:
# =============================================================================
# Cell 2: Dependencies
# =============================================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
from collections import Counter
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, 
    CSVLogger, LearningRateScheduler
)

import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_recall_fscore_support,
    precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

# Set seeds
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# GPU memory configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision for A100
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("Mixed precision (float16) enabled")
except Exception as e:
    print(f"Mixed precision not available: {e}")

print(f"TensorFlow {tf.__version__} | GPUs: {len(gpus)}")

In [ ]:
# =============================================================================
# Cell 3: Configuration
# =============================================================================

class Config:
    """Hyperparameters for DenseNet121 V2."""
    
    # Model identification
    MODEL_NAME = "DenseNet121_V2_Optimized"
    VERSION = "2.0.0"
    
    # Input configuration
    IMG_SIZE = 224  # DenseNet default
    IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
    
    # Training hyperparameters
    BATCH_SIZE = 32
    BATCH_SIZE_FINETUNE = 16  # Smaller batch for fine-tuning
    LEARNING_RATE = 1e-3      # Stage 1
    LR_STAGE2 = 3e-4          # Stage 2
    LR_STAGE3 = 1e-4          # Stage 3
    
    # Training epochs
    EPOCHS_STAGE1 = 25   # Classifier warm-up
    EPOCHS_STAGE2 = 40   # Progressive unfreezing
    EPOCHS_STAGE3 = 60   # Full fine-tuning
    
    # Architecture
    DENSE_UNITS = 2048
    DENSE_UNITS_2 = 512
    DROPOUT_RATE = 0.4
    DROPOUT_RATE_2 = 0.3
    L2_REG = 5e-4
    
    # Loss configuration
    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = 0.7  # Weight for positive class
    LABEL_SMOOTHING = 0.1
    
    # Class weighting
    MALIGNANT_WEIGHT_MULTIPLIER = 2.5
    
    # Unfreezing strategy
    FREEZE_LAYERS_STAGE2 = 150  # Keep first 150 layers frozen
    FREEZE_LAYERS_STAGE3 = 50   # Keep first 50 layers frozen
    
    # Callbacks
    EARLY_STOP_PATIENCE = 20
    LR_REDUCE_PATIENCE = 8
    LR_REDUCE_FACTOR = 0.5
    GRADIENT_CLIP_NORM = 1.0
    
    # Warmup
    USE_WARMUP = True
    WARMUP_EPOCHS = 5
    WARMUP_START_LR = 1e-6
    
    # Ensemble
    NUM_ENSEMBLE = 3
    ENSEMBLE_SEEDS = [42, 123, 456]
    
    # TTA
    USE_TTA = True
    TTA_AUGMENTATIONS = 8
    
    # Paths
    DATA_ROOT = Path("/content/drive/MyDrive/CBIS-DDSM-data") if IN_COLAB else Path("/home/tars/GoogleDrive/CBIS-DDSM-data")
    CACHE_DIR = DATA_ROOT / "preprocessed_cache_roi"
    CHECKPOINT_DIR = DATA_ROOT / "checkpoints_densenet121_v2"
    RESULTS_DIR = DATA_ROOT / "results_densenet121_v2"
    
    CLASS_NAMES = ['BENIGN', 'MALIGNANT']
    SEED = RANDOM_SEED


# Create directories
Config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Display configuration
total_epochs = Config.EPOCHS_STAGE1 + Config.EPOCHS_STAGE2 + Config.EPOCHS_STAGE3
print(f"Configuration: {Config.MODEL_NAME} v{Config.VERSION}")
print(f"  Input: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"  Dense units: {Config.DENSE_UNITS} -> {Config.DENSE_UNITS_2}")
print(f"  Training: {Config.EPOCHS_STAGE1}+{Config.EPOCHS_STAGE2}+{Config.EPOCHS_STAGE3} = {total_epochs} epochs")
print(f"  Ensemble: {Config.NUM_ENSEMBLE} models")
print(f"  Total epochs: {total_epochs * Config.NUM_ENSEMBLE}")

In [ ]:
# =============================================================================
# Cell 4: Data Loading
# =============================================================================

def load_cached_roi_data(config):
    """
    Load preprocessed ROI data from cache files.
    
    The cache contains images with CLAHE preprocessing applied.
    Values are in float32 format.
    
    Returns:
        tuple: (X_train, y_train, X_val, y_val, X_test, y_test)
    """
    cache_files = {
        'train': config.CACHE_DIR / 'train_cache.npz',
        'val': config.CACHE_DIR / 'val_cache.npz',
        'test': config.CACHE_DIR / 'test_cache.npz'
    }
    
    for name, path in cache_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Cache file not found: {path}")
    
    print("Loading ROI data from cache...")
    
    train_data = np.load(cache_files['train'])
    val_data = np.load(cache_files['val'])
    test_data = np.load(cache_files['test'])
    
    X_train = train_data['images']
    y_train = train_data['labels'].astype(np.int32)
    X_val = val_data['images']
    y_val = val_data['labels'].astype(np.int32)
    X_test = test_data['images']
    y_test = test_data['labels'].astype(np.int32)
    
    print(f"  Train: {X_train.shape[0]} samples, {X_train.nbytes / 1e9:.2f} GB")
    print(f"  Val:   {X_val.shape[0]} samples")
    print(f"  Test:  {X_test.shape[0]} samples")
    print(f"  Image range: [{X_train.min():.3f}, {X_train.max():.3f}]")
    
    return X_train, y_train, X_val, y_val, X_test, y_test


# Load data
X_train, y_train, X_val, y_val, X_test, y_test = load_cached_roi_data(Config)

# Class distribution
print("\nClass Distribution:")
for name, labels in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    benign = int((labels == 0).sum())
    malignant = int((labels == 1).sum())
    total = len(labels)
    ratio = malignant / benign if benign > 0 else 0
    print(f"  {name}: BENIGN={benign} ({benign/total*100:.1f}%), MALIGNANT={malignant} ({malignant/total*100:.1f}%), Ratio={ratio:.2f}")

In [ ]:
# =============================================================================
# Cell 5: DenseNet Preprocessing
# =============================================================================

def apply_densenet_preprocessing(images):
    """
    Apply DenseNet-specific preprocessing.
    
    DenseNet uses 'torch' mode preprocessing:
    - Scale pixels to [0, 255]
    - Subtract ImageNet mean: [123.68, 116.779, 103.939]
    - Divide by std: [58.393, 57.12, 57.375]
    
    Note: Our cached images are already in [~-0.4, ~1.4] range after CLAHE.
    We need to rescale to [0, 255] first.
    """
    # Rescale from current range to [0, 255]
    # First normalize to [0, 1], then scale to [0, 255]
    images_min = images.min()
    images_max = images.max()
    images_normalized = (images - images_min) / (images_max - images_min + 1e-8)
    images_255 = images_normalized * 255.0
    
    # Apply DenseNet preprocessing (torch mode)
    return densenet_preprocess(images_255)


def preprocess_batch_for_densenet(images):
    """
    Preprocess batch for DenseNet inference.
    Handles both training and inference.
    
    The cached images have values in approx [-0.4, 1.4] after CLAHE.
    We normalize to [0, 255] range preserving the full dynamic range,
    then apply DenseNet's torch-mode preprocessing.
    """
    # Normalize from actual range to [0, 1] first (preserving full range)
    images_min = images.min()
    images_max = images.max()
    images_normalized = (images - images_min) / (images_max - images_min + 1e-8)
    
    # Scale to [0, 255] for DenseNet preprocessing
    images_255 = images_normalized * 255.0
    
    return densenet_preprocess(images_255)


# Preprocess all data for DenseNet
print("Applying DenseNet preprocessing...")
X_train_dn = preprocess_batch_for_densenet(X_train)
X_val_dn = preprocess_batch_for_densenet(X_val)
X_test_dn = preprocess_batch_for_densenet(X_test)

print(f"  After preprocessing - range: [{X_train_dn.min():.3f}, {X_train_dn.max():.3f}]")

# Clean up original data
del X_train, X_val, X_test
gc.collect()


In [ ]:
# =============================================================================
# Cell 6: Class Weights
# =============================================================================

def compute_balanced_weights(labels, malignant_multiplier=2.5):
    """
    Compute class weights with additional emphasis on malignant class.
    
    In medical imaging, false negatives (missing cancer) are more costly
    than false positives, so we increase the weight for malignant class.
    """
    class_counts = Counter(labels.tolist())
    total = len(labels)
    
    # Balanced weights
    weights = {
        0: total / (2 * class_counts[0]),
        1: total / (2 * class_counts[1]) * malignant_multiplier
    }
    
    return weights


class_weights = compute_balanced_weights(y_train, Config.MALIGNANT_WEIGHT_MULTIPLIER)
print(f"Class Weights:")
print(f"  BENIGN (0):    {class_weights[0]:.4f}")
print(f"  MALIGNANT (1): {class_weights[1]:.4f}")

In [ ]:
# =============================================================================
# Cell 7: Data Augmentation Pipeline
# =============================================================================

@tf.function
def augment_image(image):
    """GPU-accelerated augmentation for single image."""
    # Random horizontal flip
    image = tf.image.random_flip_left_right(image)
    # Random vertical flip
    image = tf.image.random_flip_up_down(image)
    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.1)
    # Random contrast
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image


def create_dataset(X, y, config, training=True, use_reduced_batch=False):
    """
    Create tf.data pipeline with augmentation.
    
    Args:
        X: Image array (already DenseNet preprocessed)
        y: Label array
        config: Configuration object
        training: Whether to apply augmentation
        use_reduced_batch: Use smaller batch size for fine-tuning
    """
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    dataset = dataset.cache()
    
    if training:
        dataset = dataset.shuffle(min(len(X), 10000), seed=config.SEED, reshuffle_each_iteration=True)
        dataset = dataset.map(
            lambda x, y: (augment_image(x), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    
    batch_size = config.BATCH_SIZE_FINETUNE if use_reduced_batch else config.BATCH_SIZE
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset


# Create datasets
train_dataset = create_dataset(X_train_dn, y_train, Config, training=True)
val_dataset = create_dataset(X_val_dn, y_val, Config, training=False)
test_dataset = create_dataset(X_test_dn, y_test, Config, training=False)

print(f"Dataset pipelines created")
print(f"  Train batches: {len(X_train_dn) // Config.BATCH_SIZE}")
print(f"  Val batches: {len(X_val_dn) // Config.BATCH_SIZE}")

In [ ]:
# =============================================================================
# Cell 8: Focal Loss Implementation
# =============================================================================

def create_focal_loss(gamma=2.0, alpha=0.7, label_smoothing=0.1):
    """
    Focal Loss with label smoothing for handling class imbalance.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        gamma: Focusing parameter (higher = more focus on hard examples)
        alpha: Class balancing factor for positive class
        label_smoothing: Smoothing factor for soft labels
    """
    @tf.function
    def focal_loss(y_true, y_pred):
        # Apply label smoothing
        y_true = tf.cast(y_true, tf.float32)
        y_true_smooth = y_true * (1 - label_smoothing) + 0.5 * label_smoothing
        
        # Clip predictions to prevent log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        # Calculate focal weight
        p_t = y_true_smooth * y_pred + (1 - y_true_smooth) * (1 - y_pred)
        focal_weight = tf.pow(1 - p_t, gamma)
        
        # Class-balanced alpha
        alpha_t = y_true_smooth * alpha + (1 - y_true_smooth) * (1 - alpha)
        
        # Cross-entropy
        ce = -y_true_smooth * tf.math.log(y_pred) - (1 - y_true_smooth) * tf.math.log(1 - y_pred)
        
        return tf.reduce_mean(alpha_t * focal_weight * ce)
    
    return focal_loss


print(f"Focal Loss configured: gamma={Config.FOCAL_GAMMA}, alpha={Config.FOCAL_ALPHA}")

In [ ]:
# =============================================================================
# Cell 9: Model Architecture
# =============================================================================

def build_densenet121_classifier(config, seed=42):
    """
    Build DenseNet121 with custom classification head.
    
    Architecture:
    - DenseNet121 backbone (ImageNet pretrained)
    - Global Average Pooling
    - Dense(2048) -> BN -> ReLU -> Dropout(0.4)
    - Dense(512) -> BN -> ReLU -> Dropout(0.3)
    - Dense(1, sigmoid)
    
    Returns:
        tuple: (model, base_model)
    """
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
    # Input layer
    inputs = layers.Input(shape=config.IMG_SHAPE, name='input')
    
    # DenseNet121 backbone
    base_model = DenseNet121(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        pooling=None
    )
    
    # Freeze backbone initially
    base_model.trainable = False
    
    # Feature extraction
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    
    # Classification head - Layer 1
    x = layers.Dense(
        config.DENSE_UNITS,
        kernel_regularizer=regularizers.l2(config.L2_REG),
        kernel_initializer='he_normal',
        name='fc1'
    )(x)
    x = layers.BatchNormalization(name='fc1_bn')(x)
    x = layers.Activation('relu', name='fc1_relu')(x)
    x = layers.Dropout(config.DROPOUT_RATE, name='fc1_dropout')(x)
    
    # Classification head - Layer 2
    x = layers.Dense(
        config.DENSE_UNITS_2,
        kernel_regularizer=regularizers.l2(config.L2_REG),
        kernel_initializer='he_normal',
        name='fc2'
    )(x)
    x = layers.BatchNormalization(name='fc2_bn')(x)
    x = layers.Activation('relu', name='fc2_relu')(x)
    x = layers.Dropout(config.DROPOUT_RATE_2, name='fc2_dropout')(x)
    
    # Output layer (float32 for numerical stability with mixed precision)
    outputs = layers.Dense(
        1,
        activation='sigmoid',
        dtype='float32',
        name='predictions'
    )(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='DenseNet121_V2')
    
    return model, base_model


def compile_model(model, config, learning_rate):
    """Compile model with focal loss and metrics."""
    if config.USE_FOCAL_LOSS:
        loss = create_focal_loss(config.FOCAL_GAMMA, config.FOCAL_ALPHA, config.LABEL_SMOOTHING)
    else:
        loss = keras.losses.BinaryCrossentropy(label_smoothing=config.LABEL_SMOOTHING)
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate, clipnorm=config.GRADIENT_CLIP_NORM),
        loss=loss,
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )


# Build and display model
test_model, test_base = build_densenet121_classifier(Config)
print(f"Model: {test_model.name}")
print(f"  Total parameters: {test_model.count_params():,}")
print(f"  Trainable: {sum(np.prod(v.shape) for v in test_model.trainable_weights):,}")
print(f"  DenseNet121 layers: {len(test_base.layers)}")

del test_model, test_base
gc.collect()

In [ ]:
# =============================================================================
# Cell 10: Callbacks
# =============================================================================

class WarmupScheduler(keras.callbacks.Callback):
    """Linear warmup learning rate scheduler."""
    
    def __init__(self, warmup_epochs, start_lr, target_lr):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.start_lr = start_lr
        self.target_lr = target_lr
    
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
            # Keras 3 compatible learning rate assignment
            self.model.optimizer.learning_rate.assign(lr)


class MinEpochEarlyStopping(keras.callbacks.Callback):
    """Early stopping with minimum epoch requirement."""
    
    def __init__(self, monitor='val_auc', patience=20, min_epochs=15, mode='max'):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_epochs = min_epochs
        self.mode = mode
        self.wait = 0
        self.best = -np.inf if mode == 'max' else np.inf
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if epoch < self.min_epochs:
            if (self.mode == 'max' and current > self.best) or \
               (self.mode == 'min' and current < self.best):
                self.best = current
            return
        
        if (self.mode == 'max' and current > self.best) or \
           (self.mode == 'min' and current < self.best):
            self.best = current
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f"\nEarly stopping at epoch {epoch + 1}")


def create_callbacks(config, model_idx, stage):
    """Create callbacks for training stage."""
    callbacks = []
    
    # Warmup (stage 1 only)
    if stage == 1 and config.USE_WARMUP:
        callbacks.append(WarmupScheduler(
            warmup_epochs=config.WARMUP_EPOCHS,
            start_lr=config.WARMUP_START_LR,
            target_lr=config.LEARNING_RATE
        ))
    
    # Model checkpoint
    checkpoint_path = config.CHECKPOINT_DIR / f"model_{model_idx}_stage{stage}_best.keras"
    callbacks.append(ModelCheckpoint(
        str(checkpoint_path),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ))
    
    # Global best checkpoint
    global_path = config.CHECKPOINT_DIR / f"model_{model_idx}_global_best.keras"
    callbacks.append(ModelCheckpoint(
        str(global_path),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=0
    ))
    
    # Learning rate reduction
    callbacks.append(ReduceLROnPlateau(
        monitor='val_auc',
        mode='max',
        factor=config.LR_REDUCE_FACTOR,
        patience=config.LR_REDUCE_PATIENCE,
        min_lr=1e-7,
        verbose=1
    ))
    
    # Minimum epoch early stopping
    min_epochs = {
        1: config.EPOCHS_STAGE1 // 2,
        2: config.EPOCHS_STAGE2 // 2,
        3: config.EPOCHS_STAGE3 // 2
    }
    callbacks.append(MinEpochEarlyStopping(
        monitor='val_auc',
        patience=config.EARLY_STOP_PATIENCE,
        min_epochs=min_epochs[stage],
        mode='max'
    ))
    
    # CSV logger
    log_path = config.RESULTS_DIR / f"model_{model_idx}_stage{stage}_history.csv"
    callbacks.append(CSVLogger(str(log_path), append=True))
    
    return callbacks


print("Callbacks configured")

In [ ]:
# =============================================================================
# Cell 11: Progressive Training
# =============================================================================

def unfreeze_layers(base_model, num_frozen):
    """Unfreeze layers while keeping BatchNorm frozen."""
    for layer in base_model.layers[:num_frozen]:
        layer.trainable = False
    for layer in base_model.layers[num_frozen:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True


def train_single_model(config, model_idx, train_ds, val_ds, class_weights):
    """
    Train single model with 3-stage progressive unfreezing.
    
    Stage 1: Train classification head only
    Stage 2: Unfreeze top 40% of DenseNet
    Stage 3: Unfreeze top 80% of DenseNet
    
    Returns:
        dict: Combined training history
    """
    print(f"\n{'='*60}")
    print(f"Training Model {model_idx + 1}/{config.NUM_ENSEMBLE}")
    print(f"{'='*60}")
    
    seed = config.ENSEMBLE_SEEDS[model_idx]
    model, base_model = build_densenet121_classifier(config, seed=seed)
    
    all_history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': [],
                   'auc': [], 'val_auc': [], 'precision': [], 'val_precision': [],
                   'recall': [], 'val_recall': [], 'lr': []}
    
    # =================================================================
    # Stage 1: Classifier warm-up
    # =================================================================
    print(f"\nStage 1: Classifier warm-up ({config.EPOCHS_STAGE1} epochs)")
    base_model.trainable = False
    compile_model(model, config, config.LEARNING_RATE)
    
    trainable_count = sum(np.prod(v.shape) for v in model.trainable_weights)
    print(f"  Trainable parameters: {trainable_count:,}")
    
    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.EPOCHS_STAGE1,
        class_weight=class_weights,
        callbacks=create_callbacks(config, model_idx, stage=1),
        verbose=1
    )
    
    for key in all_history:
        if key in history1.history:
            all_history[key].extend(history1.history[key])
        elif key == 'lr':
            all_history['lr'].extend([float(model.optimizer.learning_rate.numpy())] * len(history1.history['loss']))
    
    # =================================================================
    # Stage 2: Progressive unfreezing (40% of backbone)
    # =================================================================
    print(f"\nStage 2: Progressive unfreezing ({config.EPOCHS_STAGE2} epochs)")
    unfreeze_layers(base_model, config.FREEZE_LAYERS_STAGE2)
    compile_model(model, config, config.LR_STAGE2)
    
    trainable_count = sum(np.prod(v.shape) for v in model.trainable_weights)
    print(f"  Trainable parameters: {trainable_count:,}")
    print(f"  Frozen layers: {config.FREEZE_LAYERS_STAGE2}")
    
    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=config.EPOCHS_STAGE2,
        class_weight=class_weights,
        callbacks=create_callbacks(config, model_idx, stage=2),
        verbose=1
    )
    
    for key in all_history:
        if key in history2.history:
            all_history[key].extend(history2.history[key])
        elif key == 'lr':
            all_history['lr'].extend([float(model.optimizer.learning_rate.numpy())] * len(history2.history['loss']))
    
    # =================================================================
    # Stage 3: Full fine-tuning (80% of backbone)
    # =================================================================
    print(f"\nStage 3: Full fine-tuning ({config.EPOCHS_STAGE3} epochs)")
    unfreeze_layers(base_model, config.FREEZE_LAYERS_STAGE3)
    compile_model(model, config, config.LR_STAGE3)
    
    trainable_count = sum(np.prod(v.shape) for v in model.trainable_weights)
    print(f"  Trainable parameters: {trainable_count:,}")
    print(f"  Frozen layers: {config.FREEZE_LAYERS_STAGE3}")
    
    # Use smaller batch size for fine-tuning
    train_ds_finetune = create_dataset(X_train_dn, y_train, config, training=True, use_reduced_batch=True)
    val_ds_finetune = create_dataset(X_val_dn, y_val, config, training=False, use_reduced_batch=True)
    
    history3 = model.fit(
        train_ds_finetune,
        validation_data=val_ds_finetune,
        epochs=config.EPOCHS_STAGE3,
        class_weight=class_weights,
        callbacks=create_callbacks(config, model_idx, stage=3),
        verbose=1
    )
    
    for key in all_history:
        if key in history3.history:
            all_history[key].extend(history3.history[key])
        elif key == 'lr':
            all_history['lr'].extend([float(model.optimizer.learning_rate.numpy())] * len(history3.history['loss']))
    
    # Load best weights
    best_checkpoint = config.CHECKPOINT_DIR / f"model_{model_idx}_global_best.keras"
    if best_checkpoint.exists():
        model.load_weights(str(best_checkpoint))
        print(f"Loaded best weights from checkpoint")
    
    # Save complete history
    history_path = config.RESULTS_DIR / f"model_{model_idx}_full_history.json"
    with open(history_path, 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in all_history.items()}, f, indent=2)
    
    del model, base_model
    gc.collect()
    keras.backend.clear_session()
    
    return all_history


print("Training function defined")

In [ ]:
# =============================================================================
# Cell 12: Train Ensemble
# =============================================================================

def check_existing_checkpoints(config):
    """Check for existing model checkpoints."""
    existing = {}
    for i in range(config.NUM_ENSEMBLE):
        checkpoint = config.CHECKPOINT_DIR / f"model_{i}_global_best.keras"
        if checkpoint.exists():
            existing[i] = checkpoint
    return existing


def train_ensemble(config, train_ds, val_ds, class_weights, skip_trained=True):
    """Train ensemble of models."""
    ensemble_histories = []
    existing = check_existing_checkpoints(config)
    
    print(f"\nTraining Ensemble: {config.NUM_ENSEMBLE} models")
    print(f"Existing checkpoints: {len(existing)}")
    
    for i in range(config.NUM_ENSEMBLE):
        if skip_trained and i in existing:
            print(f"\nSkipping model {i} - checkpoint exists")
            
            # Load existing history if available
            history_path = config.RESULTS_DIR / f"model_{i}_full_history.json"
            if history_path.exists():
                with open(history_path) as f:
                    ensemble_histories.append(json.load(f))
            else:
                ensemble_histories.append({})
            continue
        
        history = train_single_model(config, i, train_ds, val_ds, class_weights)
        ensemble_histories.append(history)
    
    print(f"\nEnsemble training complete")
    return ensemble_histories


# Training control
SKIP_TRAINED_MODELS = True

existing = check_existing_checkpoints(Config)
print(f"Existing checkpoints: {len(existing)}")

total_epochs = Config.EPOCHS_STAGE1 + Config.EPOCHS_STAGE2 + Config.EPOCHS_STAGE3
print(f"Total epochs per model: {total_epochs}")
print(f"Estimated training time: {total_epochs * Config.NUM_ENSEMBLE * 2} minutes (assuming ~2 min/epoch)")

# Train
ensemble_histories = train_ensemble(
    Config, train_dataset, val_dataset, class_weights,
    skip_trained=SKIP_TRAINED_MODELS
)

In [ ]:
# =============================================================================
# Cell 13: Test-Time Augmentation
# =============================================================================

def apply_tta(image, aug_idx):
    """
    Apply single TTA augmentation.
    
    Augmentations:
    0: Original
    1: Horizontal flip
    2: Vertical flip
    3: 90° rotation
    4: 180° rotation
    5: 270° rotation
    6: Horizontal flip + 90° rotation
    7: Vertical flip + 90° rotation
    """
    if aug_idx == 0:
        return image
    elif aug_idx == 1:
        return np.fliplr(image)
    elif aug_idx == 2:
        return np.flipud(image)
    elif aug_idx == 3:
        return np.rot90(image, k=1)
    elif aug_idx == 4:
        return np.rot90(image, k=2)
    elif aug_idx == 5:
        return np.rot90(image, k=3)
    elif aug_idx == 6:
        return np.rot90(np.fliplr(image), k=1)
    elif aug_idx == 7:
        return np.rot90(np.flipud(image), k=1)
    return image


def predict_with_tta(model, images, num_augmentations=8, batch_size=32):
    """Predict with test-time augmentation."""
    all_predictions = []
    
    for aug_idx in range(num_augmentations):
        augmented = np.array([apply_tta(img, aug_idx) for img in images])
        preds = model.predict(augmented, batch_size=batch_size, verbose=0).flatten()
        all_predictions.append(preds)
    
    return np.mean(all_predictions, axis=0)


def ensemble_predict_with_tta(models, images, num_augmentations=8, batch_size=32):
    """Ensemble prediction with TTA."""
    all_model_predictions = []
    
    for model in models:
        preds = predict_with_tta(model, images, num_augmentations, batch_size)
        all_model_predictions.append(preds)
    
    return np.mean(all_model_predictions, axis=0)


print(f"TTA configured: {Config.TTA_AUGMENTATIONS} augmentations")

In [ ]:
# =============================================================================
# Cell 14: Load Ensemble Models
# =============================================================================

def load_ensemble_models(config):
    """Load trained ensemble models from checkpoints."""
    models = []
    
    # Create focal loss with same parameters used during training
    focal_loss_fn = create_focal_loss(
        gamma=config.FOCAL_GAMMA,
        alpha=config.FOCAL_ALPHA,
        label_smoothing=config.LABEL_SMOOTHING
    )
    
    for i in range(config.NUM_ENSEMBLE):
        checkpoint = config.CHECKPOINT_DIR / f"model_{i}_global_best.keras"
        if checkpoint.exists():
            # Load without compiling to avoid custom loss deserialization issues
            model = keras.models.load_model(str(checkpoint), compile=False)
            
            # Recompile with correct loss and metrics
            model.compile(
                optimizer=Adam(learning_rate=config.LR_STAGE3, clipnorm=config.GRADIENT_CLIP_NORM),
                loss=focal_loss_fn,
                metrics=[
                    keras.metrics.BinaryAccuracy(name='accuracy'),
                    keras.metrics.AUC(name='auc'),
                    keras.metrics.Precision(name='precision'),
                    keras.metrics.Recall(name='recall')
                ]
            )
            models.append(model)
            print(f"Loaded model {i + 1}")
        else:
            print(f"Checkpoint not found: {checkpoint}")
    
    print(f"Loaded {len(models)} ensemble models")
    return models


ensemble_models = load_ensemble_models(Config)

In [ ]:
# =============================================================================
# Cell 15: Evaluation
# =============================================================================

def evaluate_ensemble(models, X_test, y_test, config, use_tta=True):
    """
    Comprehensive ensemble evaluation.
    
    Returns:
        dict: Evaluation metrics and predictions
    """
    print("Evaluating ensemble on test set...")
    
    # Individual model predictions
    individual_aucs = []
    all_predictions = []
    
    for i, model in enumerate(models):
        if use_tta:
            preds = predict_with_tta(model, X_test, config.TTA_AUGMENTATIONS)
        else:
            preds = model.predict(X_test, verbose=0).flatten()
        
        auc = roc_auc_score(y_test, preds)
        individual_aucs.append(auc)
        all_predictions.append(preds)
        print(f"  Model {i + 1} AUC: {auc:.4f}")
    
    # Ensemble predictions
    ensemble_preds = np.mean(all_predictions, axis=0)
    ensemble_binary = (ensemble_preds > 0.5).astype(int)
    
    # Metrics at threshold 0.5
    ensemble_auc = roc_auc_score(y_test, ensemble_preds)
    accuracy = accuracy_score(y_test, ensemble_binary)
    precision = precision_score(y_test, ensemble_binary, zero_division=0)
    recall = recall_score(y_test, ensemble_binary, zero_division=0)
    f1 = f1_score(y_test, ensemble_binary, zero_division=0)
    
    cm = confusion_matrix(y_test, ensemble_binary)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Find optimal threshold (Youden's J)
    fpr, tpr, thresholds = roc_curve(y_test, ensemble_preds)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    # Metrics at optimal threshold
    optimal_binary = (ensemble_preds > optimal_threshold).astype(int)
    optimal_accuracy = accuracy_score(y_test, optimal_binary)
    optimal_precision = precision_score(y_test, optimal_binary, zero_division=0)
    optimal_recall = recall_score(y_test, optimal_binary, zero_division=0)
    optimal_f1 = f1_score(y_test, optimal_binary, zero_division=0)
    
    cm_opt = confusion_matrix(y_test, optimal_binary)
    tn_opt, fp_opt, fn_opt, tp_opt = cm_opt.ravel()
    optimal_specificity = tn_opt / (tn_opt + fp_opt) if (tn_opt + fp_opt) > 0 else 0
    
    results = {
        'ensemble_auc': float(ensemble_auc),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'sensitivity': float(recall),
        'specificity': float(specificity),
        'f1_score': float(f1),
        'optimal_threshold': float(optimal_threshold),
        'optimal_accuracy': float(optimal_accuracy),
        'optimal_precision': float(optimal_precision),
        'optimal_recall': float(optimal_recall),
        'optimal_sensitivity': float(optimal_recall),
        'optimal_specificity': float(optimal_specificity),
        'optimal_f1': float(optimal_f1),
        'individual_aucs': [float(a) for a in individual_aucs],
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'optimal_confusion_matrix': {'tn': int(tn_opt), 'fp': int(fp_opt), 'fn': int(fn_opt), 'tp': int(tp_opt)},
        'predictions': ensemble_preds.tolist(),
        'true_labels': y_test.tolist(),
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist()
    }
    
    return results


# Evaluate
if ensemble_models and len(ensemble_models) > 0:
    results = evaluate_ensemble(ensemble_models, X_test_dn, y_test, Config, use_tta=Config.USE_TTA)
    
    print("\n" + "="*60)
    print("TEST SET RESULTS")
    print("="*60)
    print(f"\nThreshold = 0.5:")
    print(f"  AUC:         {results['ensemble_auc']:.4f}")
    print(f"  Accuracy:    {results['accuracy']:.4f}")
    print(f"  Sensitivity: {results['sensitivity']:.4f}")
    print(f"  Specificity: {results['specificity']:.4f}")
    print(f"  F1 Score:    {results['f1_score']:.4f}")
    print(f"\nOptimal Threshold = {results['optimal_threshold']:.3f}:")
    print(f"  Accuracy:    {results['optimal_accuracy']:.4f}")
    print(f"  Sensitivity: {results['optimal_sensitivity']:.4f}")
    print(f"  Specificity: {results['optimal_specificity']:.4f}")
    print(f"  F1 Score:    {results['optimal_f1']:.4f}")
else:
    print("No models found for evaluation")

In [ ]:
# =============================================================================
# Cell 16: Visualization
# =============================================================================

def plot_results(results, histories, config):
    """Generate evaluation plots."""
    fig = plt.figure(figsize=(16, 12))
    
    # ROC Curve
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(results['fpr'], results['tpr'], 'b-', linewidth=2,
             label=f"AUC = {results['ensemble_auc']:.4f}")
    ax1.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax1.fill_between(results['fpr'], results['tpr'], alpha=0.2)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)
    
    # Confusion Matrix
    ax2 = fig.add_subplot(2, 2, 2)
    cm = np.array([
        [results['confusion_matrix']['tn'], results['confusion_matrix']['fp']],
        [results['confusion_matrix']['fn'], results['confusion_matrix']['tp']]
    ])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
                xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES)
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('Actual')
    ax2.set_title('Confusion Matrix (threshold=0.5)')
    
    # Training AUC
    ax3 = fig.add_subplot(2, 2, 3)
    for i, h in enumerate(histories):
        if h and 'auc' in h:
            ax3.plot(h['auc'], label=f'M{i} train', alpha=0.7)
            ax3.plot(h['val_auc'], '--', label=f'M{i} val', alpha=0.7)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('AUC')
    ax3.set_title('Training AUC')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    
    # Training Loss
    ax4 = fig.add_subplot(2, 2, 4)
    for i, h in enumerate(histories):
        if h and 'loss' in h:
            ax4.plot(h['loss'], label=f'M{i} train', alpha=0.7)
            ax4.plot(h['val_loss'], '--', label=f'M{i} val', alpha=0.7)
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Loss')
    ax4.set_title('Training Loss')
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / 'evaluation_plots.png', dpi=150, bbox_inches='tight')
    plt.show()


if 'results' in globals() and 'ensemble_histories' in globals() and results and ensemble_histories:
    plot_results(results, ensemble_histories, Config)
else:
    print("Skipping visualization - results or histories not available")

In [ ]:
# =============================================================================
# Cell 17: Save Results
# =============================================================================

def save_results(results, config):
    """Save final results to JSON."""
    final_results = {
        'model_name': config.MODEL_NAME,
        'version': config.VERSION,
        'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
        'configuration': {
            'img_size': config.IMG_SIZE,
            'batch_size': config.BATCH_SIZE,
            'dense_units': [config.DENSE_UNITS, config.DENSE_UNITS_2],
            'dropout_rates': [config.DROPOUT_RATE, config.DROPOUT_RATE_2],
            'epochs': f"{config.EPOCHS_STAGE1}+{config.EPOCHS_STAGE2}+{config.EPOCHS_STAGE3}",
            'ensemble_size': config.NUM_ENSEMBLE,
            'focal_loss': config.USE_FOCAL_LOSS,
            'tta_augmentations': config.TTA_AUGMENTATIONS,
            'l2_reg': config.L2_REG
        },
        'metrics': {
            'ensemble_auc': results['ensemble_auc'],
            'accuracy': results['accuracy'],
            'precision': results['precision'],
            'recall': results['recall'],
            'sensitivity': results['sensitivity'],
            'specificity': results['specificity'],
            'f1_score': results['f1_score'],
            'individual_aucs': results['individual_aucs']
        },
        'optimal_metrics': {
            'threshold': results['optimal_threshold'],
            'accuracy': results['optimal_accuracy'],
            'sensitivity': results['optimal_sensitivity'],
            'specificity': results['optimal_specificity'],
            'f1_score': results['optimal_f1']
        },
        'confusion_matrix': results['confusion_matrix'],
        'optimal_confusion_matrix': results['optimal_confusion_matrix'],
        'dataset': {
            'train': len(y_train),
            'val': len(y_val),
            'test': len(y_test)
        }
    }
    
    results_path = config.RESULTS_DIR / 'densenet121_v2_results.json'
    with open(results_path, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    print(f"Results saved: {results_path}")
    return final_results


if 'results' in globals() and results:
    final_results = save_results(results, Config)
else:
    print("Skipping save - no results available")

In [ ]:
# =============================================================================
# Cell 18: Summary
# =============================================================================

print(f"""
{'='*70}
DenseNet121 V2 Training Complete
{'='*70}

Architecture:
  - DenseNet121 backbone (ImageNet pretrained)
  - Dense({Config.DENSE_UNITS}) -> BN -> ReLU -> Dropout({Config.DROPOUT_RATE})
  - Dense({Config.DENSE_UNITS_2}) -> BN -> ReLU -> Dropout({Config.DROPOUT_RATE_2})
  - Dense(1, sigmoid)

Preprocessing:
  - DenseNet preprocess_input (torch mode)
  - ImageNet normalization applied

Training:
  - Stage 1: {Config.EPOCHS_STAGE1} epochs (classifier only)
  - Stage 2: {Config.EPOCHS_STAGE2} epochs (unfreeze 40%)
  - Stage 3: {Config.EPOCHS_STAGE3} epochs (unfreeze 80%)
  - Ensemble: {Config.NUM_ENSEMBLE} models
  - TTA: {Config.TTA_AUGMENTATIONS} augmentations

Results:
  - Checkpoints: {Config.CHECKPOINT_DIR}
  - Results: {Config.RESULTS_DIR}

Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*70}
""")